# Atlas Distribuidora Analytics — Análise Exploratória\n\nEste notebook documenta a EDA que conecta os dados sintéticos às decisões de negócio e ao dashboard Power BI.\n\n**Perguntas centrais:** evolução do faturamento, margem, concentração de clientes, mix de produtos, performance comercial, metas e estoque.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nDATA_DIR = Path('../data/generated')\nif not DATA_DIR.exists():\n    DATA_DIR = Path('data/generated')\n\nDATA_DIR

## 1. Carregamento e construção da base analítica

In [ ]:
pedido = pd.read_csv(DATA_DIR / 'pedido.csv', parse_dates=['data_pedido', 'data_faturamento'])\nitem = pd.read_csv(DATA_DIR / 'pedido_item.csv')\ncliente = pd.read_csv(DATA_DIR / 'cliente.csv')\nproduto = pd.read_csv(DATA_DIR / 'produto.csv')\ncategoria = pd.read_csv(DATA_DIR / 'categoria.csv')\nrepresentante = pd.read_csv(DATA_DIR / 'representante.csv')\nmeta = pd.read_csv(DATA_DIR / 'meta_representante.csv', parse_dates=['competencia'])\nestoque = pd.read_csv(DATA_DIR / 'estoque.csv')\n\nvendas = (\n    item.merge(\n        pedido.loc[pedido['status'].isin(['FATURADO', 'ENTREGUE']) & pedido['data_faturamento'].notna()],\n        on='id_pedido', how='inner', validate='many_to_one'\n    )\n    .merge(produto[['id_produto', 'sku', 'descricao', 'id_categoria']], on='id_produto', how='left')\n    .merge(categoria, on='id_categoria', how='left')\n)\n\nvendas['valor_bruto'] = vendas['quantidade'] * vendas['preco_unitario']\nvendas['faturamento'] = vendas['valor_bruto'] - vendas['desconto_item']\nvendas['custo_total'] = vendas['quantidade'] * vendas['custo_unitario']\nvendas['margem_bruta'] = vendas['faturamento'] - vendas['custo_total']\nvendas['competencia'] = vendas['data_faturamento'].dt.to_period('M').dt.to_timestamp()\n\nvendas.head()

## 2. KPIs executivos

In [ ]:
faturamento = vendas['faturamento'].sum()\nmargem = vendas['margem_bruta'].sum()\npedidos = vendas['id_pedido'].nunique()\n\npd.Series({\n    'Faturamento': faturamento,\n    'Margem Bruta': margem,\n    'Margem %': margem / faturamento,\n    'Pedidos': pedidos,\n    'Ticket Médio': faturamento / pedidos,\n    'Clientes Compradores': vendas['id_cliente'].nunique(),\n})

## 3. Evolução mensal

In [ ]:
mensal = (\n    vendas.groupby('competencia', as_index=False)\n    .agg(faturamento=('faturamento', 'sum'), margem=('margem_bruta', 'sum'))\n    .sort_values('competencia')\n)\nmensal['crescimento_mom'] = mensal['faturamento'].pct_change()\nmensal

In [ ]:
ax = mensal.plot(x='competencia', y='faturamento', figsize=(11, 5), marker='o', legend=False)\nax.set_title('Evolução mensal do faturamento')\nax.set_xlabel('')\nax.grid(alpha=.25);

## 4. Produtos e categorias

In [ ]:
performance_categoria = (\n    vendas.groupby('nome', as_index=False)\n    .agg(faturamento=('faturamento', 'sum'), margem=('margem_bruta', 'sum'), quantidade=('quantidade', 'sum'))\n    .sort_values('faturamento', ascending=False)\n)\nperformance_categoria['margem_pct'] = performance_categoria['margem'] / performance_categoria['faturamento']\nperformance_categoria

## 5. Clientes e concentração de receita

In [ ]:
clientes = (\n    vendas.groupby('id_cliente', as_index=False)\n    .agg(faturamento=('faturamento', 'sum'), pedidos=('id_pedido', 'nunique'))\n    .sort_values('faturamento', ascending=False)\n)\nclientes['participacao'] = clientes['faturamento'] / clientes['faturamento'].sum()\nclientes['participacao_acumulada'] = clientes['participacao'].cumsum()\nclientes.head(10)

## 6. Performance comercial e metas

In [ ]:
vendas_rep_mes = vendas.groupby(['competencia', 'id_representante'], as_index=False)['faturamento'].sum()\nmetas = meta.groupby(['competencia', 'id_representante'], as_index=False)['valor_meta'].sum()\nperformance = vendas_rep_mes.merge(metas, on=['competencia', 'id_representante'], how='outer').fillna(0)\nperformance['atingimento'] = performance['faturamento'] / performance['valor_meta'].replace(0, pd.NA)\nperformance.sort_values('atingimento', ascending=False).head(20)

## 7. Estoque

In [ ]:
estoque = estoque.merge(produto[['id_produto', 'descricao', 'custo_atual']], on='id_produto', how='left')\nestoque['status'] = 'ADEQUADO'\nestoque.loc[estoque['estoque_atual'] < estoque['estoque_minimo'], 'status'] = 'ABAIXO_MINIMO'\nestoque.loc[estoque['estoque_atual'] > estoque['estoque_maximo'], 'status'] = 'ACIMA_MAXIMO'\nestoque['valor_estoque'] = estoque['estoque_atual'] * estoque['custo_atual']\nestoque.groupby('status').agg(posicoes=('id_estoque', 'count'), valor_estoque=('valor_estoque', 'sum'))

## 8. Do insight ao dashboard\n\nAs conclusões desta EDA devem ser transformadas em visuais apenas quando ajudarem a responder uma pergunta de negócio. O próximo passo é validar os resultados contra SQL e implementar as cinco páginas definidas na especificação do Power BI.